# Phase 1: Customer Churn Data Cleaning & Feature Engineering

## 1. Executive Summary & Context
This notebook establishes the foundational dataset prep for the Customer Churn & Retention Analysis project.
Raw data is ingested from `data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv`, audited for missing values and type inconsistencies, enriched with operational feature buckets, and exported to `data/processed/churn_clean.csv`.

### Business Analyst Decisions:
1. **`TotalCharges` Imputation:** 11 customers with `tenure == 0` contain blank string entries (`" "`). These represent newly onboarded accounts before their first billing cycle. `TotalCharges` is coerced to numeric and imputed as `0.0` to preserve the rows while accurately representing historical spend.
2. **Feature Engineering:**
   - `tenure_bucket`: 4 tenure cohorts (`0-12 Months`, `13-24 Months`, `25-48 Months`, `49+ Months`).
   - `num_services`: Integer count (0 to 6) of subscribed add-on internet services (`OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`).
   - `is_high_value`: Binary flag (1 if `MonthlyCharges` $\ge$ 75th percentile threshold, 0 otherwise).


In [7]:
import os
import pandas as pd
import numpy as np

# Load raw dataset
raw_path = '../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv'
df = pd.read_csv(raw_path)

print(f"Raw dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

Raw dataset shape: 7043 rows, 21 columns


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 2. Data Cleaning & Type Conversion

### Resolving `TotalCharges` Quality Issue
Inspect blank string values in `TotalCharges` and coerce to float64.

In [8]:
# Identify blank entries
blank_rows = df[df['TotalCharges'].astype(str).str.strip() == '']
print(f"Blank TotalCharges count: {len(blank_rows)}")
print("Tenure values of blank rows:", blank_rows['tenure'].unique())

# Coerce to numeric float64 and fill missing tenure==0 rows with 0.0
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].astype(str).str.strip(), errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0.0)

print("Processed TotalCharges dtype:", df['TotalCharges'].dtype)
print("Null count in TotalCharges:", df['TotalCharges'].isnull().sum())

Blank TotalCharges count: 11
Tenure values of blank rows: [0]
Processed TotalCharges dtype: float64
Null count in TotalCharges: 0


## 3. Operational Feature Engineering

### Feature 1: `tenure_bucket`

In [9]:
def get_tenure_bucket(t):
    if t <= 12:
        return '0-12 Months'
    elif t <= 24:
        return '13-24 Months'
    elif t <= 48:
        return '25-48 Months'
    else:
        return '49+ Months'

df['tenure_bucket'] = df['tenure'].apply(get_tenure_bucket)
print("tenure_bucket distribution:")
print(df['tenure_bucket'].value_counts())

tenure_bucket distribution:
tenure_bucket
49+ Months      2239
0-12 Months     2186
25-48 Months    1594
13-24 Months    1024
Name: count, dtype: int64


### Feature 2: `num_services`

In [10]:
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df['num_services'] = (df[service_cols] == 'Yes').sum(axis=1)

print("num_services distribution:")
print(df['num_services'].value_counts().sort_index())

num_services distribution:
num_services
0    2219
1     966
2    1033
3    1118
4     852
5     571
6     284
Name: count, dtype: int64


### Feature 3: `is_high_value`

In [11]:
p75 = df['MonthlyCharges'].quantile(0.75)
df['is_high_value'] = (df['MonthlyCharges'] >= p75).astype(int)

print(f"MonthlyCharges 75th Percentile Threshold: ${p75:.2f}")
print("is_high_value counts:")
print(df['is_high_value'].value_counts())

MonthlyCharges 75th Percentile Threshold: $89.85
is_high_value counts:
is_high_value
0    5272
1    1771
Name: count, dtype: int64


## 4. Export Cleaned Dataset

In [12]:
output_path = '../data/processed/churn_clean.csv'
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df.to_csv(output_path, index=False)
print(f"Successfully saved clean dataset ({df.shape[0]} rows, {df.shape[1]} cols) to {output_path}.")

Successfully saved clean dataset (7043 rows, 24 cols) to ../data/processed/churn_clean.csv.
